In [2609]:
import glob, os
import numpy as np
import pyxdf, mne

In [2610]:
# === ディレクトリパスを指定 ===
base_dir = r"E:\EEG_B4\sub-P001\ses-S002\eeg"

# === run-001 ～ run-003 の .xdf を自動抽出 ===
xdf_files = sorted(glob.glob(os.path.join(base_dir, "*run-00[1-3]*_eeg.xdf")))
print(f"{len(xdf_files)} files found:")
for f in xdf_files:
    print(" -", os.path.basename(f))

3 files found:
 - sub-P001_ses-S002_task-MI_run-001_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-002_eeg.xdf
 - sub-P001_ses-S002_task-MI_run-003_eeg.xdf


In [2611]:
# === 共通設定 ===
CH_LABELS = ["FC1","FC2","C3","C4","CP1","CP2","P3","P4"]
SFREQ = 250
EVENT_ID = {"left": 1, "right": 2}
TMIN, TMAX = -0.5, 4.5
# === 前処理の設定 ===
APPLY_FILTERS   = True        # 学習時にフィルタしていたなら True
NOTCH_HZ        = 50.0        # 日本の商用電源
BANDPASS        = (1.0, 40.0) # MIなら 8–30Hz などに変えてもOK（学習時と合わせる）
REJECT_PTP_UV   = None         # 例: 150 ならピークtoピーク150µV超を自動除外。使わないなら None
DO_ZSCORE       = True        # 学習時の正規化に合わせる（epoch内z-scoreが無難）

In [2612]:
all_x, all_y = [], []
for path in xdf_files:
    print(f"[INFO] Loading {os.path.basename(path)}")

    streams, header = pyxdf.load_xdf(
    path,
    synchronize_clocks=False,   # ★時計同期を無効化
    dejitter_timestamps=False,  # ★ジッタ補正も無効化
)


    eeg_stream = [s for s in streams if s["info"]["type"][0].lower() == "eeg"][0]
    data = eeg_stream["time_series"].T                 # (n_ch, n_samples) μV
    ts_eeg = eeg_stream["time_stamps"]

    marker_stream = [s for s in streams if s["info"]["type"][0].lower() == "markers"][0]
    markers = marker_stream["time_series"]
    ts_mrk = marker_stream["time_stamps"]

    # --- Raw生成（μV→V）
    info = mne.create_info(ch_names=CH_LABELS, sfreq=SFREQ, ch_types="eeg")
    raw = mne.io.RawArray(data * 1e-6, info, verbose=False)

    # --- フィルタ（学習時と同じに！）
    if APPLY_FILTERS:
        raw.notch_filter(NOTCH_HZ, fir_design="firwin", verbose=False)
        raw.filter(BANDPASS[0], BANDPASS[1], fir_design="firwin", verbose=False)

    # --- イベント配列
    events = []
    for lab, t in zip(markers, ts_mrk):
        label = lab[0] if isinstance(lab, (list, tuple, np.ndarray)) else lab
        if label not in EVENT_ID:
            continue
        fs = raw.info["sfreq"]
        # EEGタイムスタンプ基準でサンプルに変換
        sample = int((t - ts_eeg[0]) * fs)
        events.append([sample, 0, EVENT_ID[label]])
    events = np.array(events, dtype=int)
    if len(events) == 0:
        print(f"[WARN] no events in {os.path.basename(path)}")
        continue

    # --- （任意）アーティファクト除外しながらEpochs作成
    reject = None if REJECT_PTP_UV is None else dict(eeg=REJECT_PTP_UV * 1e-6)
    epochs = mne.Epochs(raw, events, event_id=EVENT_ID, tmin=TMIN, tmax=TMAX,
                        baseline=(-0.5,0.0), preload=True, reject=reject, verbose="ERROR")

    x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
    y = epochs.events[:, 2] - 1

    all_x.append(x)
    all_y.append(y)


[INFO] Loading sub-P001_ses-S002_task-MI_run-001_eeg.xdf
[INFO] Loading sub-P001_ses-S002_task-MI_run-002_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_21428\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_21428\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


[INFO] Loading sub-P001_ses-S002_task-MI_run-003_eeg.xdf


C:\Users\kgtk0\AppData\Local\Temp\ipykernel_21428\2722679931.py:49: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  x = epochs.get_data()                # (n_trials, n_ch, n_times)  V単位


In [2613]:
fs = 250
tmin, tmax = -0.5, 4.5
expected_T = int(round((tmax - tmin) * fs)) + 1 

all_x_fixed, all_y_fixed, all_groups = [], [], []
for file_idx, (x, y) in enumerate(zip(all_x, all_y)):
    T = x.shape[2]
    if T >= expected_T:
        
        x = x[..., :expected_T]
    else:
         #まれに短い場合だけスキップ or 末尾パディング（必要なら）
        #pad = expected_T - T
        #x = np.pad(x, ((0,0),(0,0),(0,pad)), mode='edge')
        pass
    all_x_fixed.append(x)
    all_y_fixed.append(y)
    all_groups.append(np.full(len(y), file_idx, dtype=int))
x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)
groups = np.concatenate(all_groups, axis=0)
print("all:", x_all.shape, y_all.shape)

all: (66, 8, 1251) (66,)


In [2614]:
# --- Load source z-score stats (robust reshape) ---
src_stats_path = r"C:\Users\kgtk0\Documents\Python\UDA_FT\zscore_stats.npz"
stats = np.load(src_stats_path)

# どのキー名でも拾えるように
mean_tr = stats.get("mean",  stats.get("mean_ch"))
std_tr  = stats.get("std",   stats.get("std_ch"))

mean_tr = np.asarray(mean_tr)
std_tr  = np.asarray(std_tr)

C = x_all.shape[1]  # 8 のはず

def _to_1C1(a, C):
    # (C,) or (C,1) or (1,C,1) を (1,C,1) にそろえる
    if a.ndim == 1 and a.shape[0] == C:
        a = a.reshape(1, C, 1)
    elif a.ndim == 2 and a.shape == (C, 1):
        a = a.reshape(1, C, 1)
    elif a.ndim == 3 and a.shape == (1, C, 1):
        pass
    else:
        raise ValueError(f"Unexpected stats shape {a.shape}; expected (C,), (C,1) or (1,C,1)")
    return a

mean_tr = _to_1C1(mean_tr, C)
std_tr  = _to_1C1(std_tr,  C)

# 念のため 0割防止
eps = 1e-8
std_tr = std_tr + eps

# --- Apply z-score using source stats ---
x_all = ((x_all - mean_tr) / std_tr).astype("float32")

#print("Shapes :", x_all.shape, mean_tr.shape, std_tr.shape)


In [2615]:
# --- 準備 ---
rng = np.random.default_rng(4)                 # 再現性
idx_all = np.arange(len(y_all))                  # 0..N-1
target_n = int(round(len(idx_all) * 0.2))        # 132→27

# --- 各 group×class から約20%をTestに回す（最低1件ルール） ---
test_idx_parts = []
for g in np.unique(groups):
    g_mask = (groups == g)
    g_idx = idx_all[g_mask]
    y_g   = y_all[g_mask]
    for cls in np.unique(y_g):
        cls_idx = g_idx[y_g == cls]
        if len(cls_idx) == 0:
            continue
        n_take = max(1, int(round(len(cls_idx) * 0.2)))
        pick = rng.choice(cls_idx, size=min(n_take, len(cls_idx)), replace=False)
        test_idx_parts.append(pick)

test_idx = np.sort(np.concatenate(test_idx_parts))
# --- ちょうど target_n (=27) に合わせて微調整 ---
if len(test_idx) > target_n:
    # 余分をランダムに削る（層化はすでに担保されているので微調整は一様でOK）
    drop = rng.choice(test_idx, size=(len(test_idx) - target_n), replace=False)
    test_idx = np.sort(np.setdiff1d(test_idx, drop))
elif len(test_idx) < target_n:
    # 足りない分は残りから層化っぽく追加
    remaining = np.setdiff1d(idx_all, test_idx)
    need = target_n - len(test_idx)
    # クラスバランスを軽く意識：各クラスから交互に補充
    add_list = []
    classes = np.unique(y_all)
    while need > 0:
        for cls in classes:
            cand = remaining[y_all[remaining] == cls]
            if len(cand) > 0:
                pick = rng.choice(cand, size=1, replace=False)
                add_list.append(pick)
                remaining = remaining[remaining != pick]
                need -= 1
                if need == 0:
                    break
    test_idx = np.sort(np.concatenate([test_idx] + add_list))

# --- Pool は Test 以外 ---
pool_idx = np.setdiff1d(idx_all, test_idx)

print(f"Total={len(idx_all)}  Test={len(test_idx)}  Pool={len(pool_idx)}")
# 参考表示：各group×classの内訳
for g in np.unique(groups):
    for cls in np.unique(y_all):
        tg = test_idx[(groups[test_idx]==g) & (y_all[test_idx]==cls)]
        pg = pool_idx[(groups[pool_idx]==g) & (y_all[pool_idx]==cls)]
        print(f"group {g} cls {cls}: test={len(tg):2d}, pool={len(pg):2d}")


Total=66  Test=13  Pool=53
group 0 cls 0: test= 3, pool= 8
group 0 cls 1: test= 2, pool= 9
group 1 cls 0: test= 2, pool= 9
group 1 cls 1: test= 2, pool= 9
group 2 cls 0: test= 2, pool= 9
group 2 cls 1: test= 2, pool= 9


In [2616]:
# --- 3) FT用: Poolから各クラス15〜20試行（合計~30–40）を層化抽出 ---
ft_idx_parts = []
for cls in np.unique(y_all):
    cls_pool = pool_idx[y_all[pool_idx] == cls]
    if len(cls_pool) == 0:
        continue
    n_ft = min(20, max(15, int(round(len(cls_pool) * 0.35))))  # 目安
    n_ft = min(n_ft, len(cls_pool))  # ★安全ガード
    pick = rng.choice(cls_pool, size=n_ft, replace=False)
    ft_idx_parts.append(pick)

ft_idx = np.sort(np.concatenate(ft_idx_parts))
print("FT total:", len(ft_idx))

# --- （任意推奨）FT内をtrain/valに8:2層化分割（早期停止用） ---
ft_train_idx_parts, ft_val_idx_parts = [], []
for cls in np.unique(y_all[ft_idx]):
    cls_ft = ft_idx[y_all[ft_idx] == cls]
    if len(cls_ft) == 1:
        ft_train_idx_parts.append(cls_ft)
        continue
    n_val = max(1, int(round(len(cls_ft) * 0.2)))
    val_pick = rng.choice(cls_ft, size=n_val, replace=False)
    train_pick = np.setdiff1d(cls_ft, val_pick)
    ft_val_idx_parts.append(val_pick)
    ft_train_idx_parts.append(train_pick)

ft_train_idx = np.sort(np.concatenate(ft_train_idx_parts))
ft_val_idx   = np.sort(np.concatenate(ft_val_idx_parts))

print(f"FT train={len(ft_train_idx)}  FT val={len(ft_val_idx)}")


FT total: 30
FT train=24  FT val=6


In [2617]:
# --- 4) 出力を保存（UDA=Pool全体は無ラベル／FTはラベルあり／Testは完全ホールドアウト） ---
out_path = r"E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz"

np.savez(out_path,
    # ---- 本体データ ----
    x_pool = x_all[pool_idx].astype("float32"),     # UDA無ラベル投入
    y_pool = y_all[pool_idx].astype("int64"),       # 参照用（UDAでは未使用）
    grp_pool = groups[pool_idx].astype("int64"),

    x_ft   = x_all[ft_idx].astype("float32"),       # FTに使う分（ラベルあり）
    y_ft   = y_all[ft_idx].astype("int64"),

    x_test = x_all[test_idx].astype("float32"),     # 最終評価のみ
    y_test = y_all[test_idx].astype("int64"),
    grp_test = groups[test_idx].astype("int64"),

    # ---- インデックス（再現性・追跡用）----
    pool_idx = pool_idx.astype("int64"),
    ft_idx   = ft_idx.astype("int64"),
    test_idx = test_idx.astype("int64"),

    # （任意だが推奨）FTのtrain/valを分けて保存：早期停止にそのまま使える
    ft_train_idx = ft_train_idx.astype("int64"),
    ft_val_idx   = ft_val_idx.astype("int64"),

    # ---- 前処理メタデータ（後で迷子にならないために）----
    ch_labels = np.array(CH_LABELS),
    sfreq = np.array([SFREQ], dtype="float32"),
    tmin  = np.array([TMIN],  dtype="float32"),
    tmax  = np.array([TMAX],  dtype="float32"),
    notch_hz = np.array([NOTCH_HZ], dtype="float32"),
    bandpass = np.array(BANDPASS, dtype="float32"),
    apply_filters = np.array([APPLY_FILTERS], dtype="int8"),

    # 使用したソース統計の形（確認用）
    src_mean_shape = np.array(mean_tr.shape),
    src_std_shape  = np.array(std_tr.shape),

    # 乱数seed（再現性メモ）
    rng_seed = np.array([14], dtype="int64"),
)

print(
    "Saved:", out_path,
    "\n  pool:", len(pool_idx),
    "\n  FT  :", len(ft_idx), f"(train={len(ft_train_idx)}, val={len(ft_val_idx)})",
    "\n  test:", len(test_idx),
)


Saved: E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz 
  pool: 53 
  FT  : 30 (train=24, val=6) 
  test: 13


In [2618]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- パス ----
SRC_NPZ = r"E:\EEG_B4\UDADataset\source_physionet_prepared.npz"   # x_train, y_train（ラベルあり）
TGT_NPZ = r"E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz"   # x_pool（無ラベル）, x_test/y_test

# ---- ソース（PhysioNet, ラベルあり） ----
src = np.load(SRC_NPZ)
x_train_src = torch.from_numpy(src["x_train"])   # shape: (Ns, C, T) float32
y_train_src = torch.from_numpy(src["y_train"])   # shape: (Ns,) int64
x_test_src = torch.from_numpy(src["x_test"])
y_test_src = torch.from_numpy(src["y_test"])
src_loader = DataLoader(TensorDataset(x_train_src, y_train_src), batch_size=128, shuffle=True, drop_last=True)

# ---- ターゲット（OpenBCI, 無ラベル） ----
tgt = np.load(TGT_NPZ)
x_pool = torch.from_numpy(tgt["x_pool"])   # shape: (Nt, C, T) float32

class TgtPool(Dataset):
    def __init__(self, X): self.X = X
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

Bt = 64  # CORAL/MMDの安定のためターゲットBtは>=32推奨
tgt_ds = TgtPool(x_pool)
tgt_sampler = RandomSampler(tgt_ds, replacement=True, num_samples=max(len(tgt_ds), 5000))
tgt_loader = DataLoader(tgt_ds, batch_size=Bt, sampler=tgt_sampler, drop_last=True)

# ---- テスト（完全ホールドアウト） ----
x_test = torch.from_numpy(tgt["x_test"])  # (Ntest, C, T)
y_test = torch.from_numpy(tgt["y_test"])  # (Ntest,)
C, T = x_pool.shape[1], x_pool.shape[2]
print("src:", x_train_src.shape, "tgt_pool:", x_pool.shape, "test:", x_test.shape)


device: cuda
src: torch.Size([3756, 8, 1251]) tgt_pool: torch.Size([53, 8, 1251]) test: torch.Size([13, 8, 1251])


In [2619]:
def coral_loss(Zs, Zt, eps=1e-6):
    Zs = Zs - Zs.mean(0, keepdim=True)
    Zt = Zt - Zt.mean(0, keepdim=True)
    d = Zs.size(1)
    Cs = (Zs.T @ Zs) / (Zs.size(0)-1 + eps)
    Ct = (Zt.T @ Zt) / (Zt.size(0)-1 + eps)
    return ((Cs - Ct).pow(2).sum()) / (4.0 * d * d)

def mmd_rbf(Zs, Zt):
    with torch.no_grad():
        allz = torch.cat([Zs, Zt], 0)
        pd = torch.cdist(allz, allz, p=2)
        med = torch.median(pd[pd>0]).clamp_min(1e-6)
        gammas = [1/(2*(med**2)), 1/(2*((2*med)**2)), 1/(2*((0.5*med)**2))]
    def k(x,y,gs):
        xx=(x*x).sum(1,keepdim=True); yy=(y*y).sum(1,keepdim=True)
        dist = xx + yy.T - 2*(x@y.T)
        return sum(torch.exp(-g*dist) for g in gs)
    Kxx, Kyy, Kxy = k(Zs,Zs,gammas), k(Zt,Zt,gammas), k(Zs,Zt,gammas)
    m,n = Zs.size(0), Zt.size(0)
    return (Kxx.sum()-Kxx.diag().sum())/(m*(m-1)+1e-6) \
         + (Kyy.sum()-Kyy.diag().sum())/(n*(n-1)+1e-6) \
         - 2*Kxy.mean()


In [2620]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=C, samples=T):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

        # 便利: 特徴次元を属性に持っておく（UDAで使う）
        self.feature_dim = self.classify.in_features

    def _get_flattened_size(self, channels, samples):
        with torch.no_grad():
            x = torch.zeros(1, 1, channels, samples)  # (B,1,C,T)
            z = self.firstconv(x)
            z = self.depthwiseConv(z)
            z = self.separableConv(z)
            return z.numel()

    # ここから追加 ---------
    def extract_features(self, x):
        """(B,C,T) or (B,1,C,T) -> (B,d) の特徴ベクトル"""
        if x.dim() == 3:            # (B,C,T) なら (B,1,C,T) に
            x = x.unsqueeze(1)
        z = self.firstconv(x)
        z = self.depthwiseConv(z)
        z = self.separableConv(z)
        z = z.view(z.size(0), -1)   # Flatten -> (B,d)
        return z

    def forward_with_features(self, x):
        Z = self.extract_features(x)         # (B,d)
        logits = self.classify(Z)            # (B,num_classes)
        return Z, logits

    def forward(self, x):
        return self.forward_with_features(x)[1]
    # ここまで追加 ---------


In [2621]:
# === モデル準備 ===
model = EEGNet(num_classes=2, channels=C, samples=T).to(device)  # ←EEGNetはあなたの実装名に置換
ce = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
# 0.01 → 0.001(20ep) → 0.0001(50ep)
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[20,50], gamma=0.1)

def one_epoch(stage, lam_coral, lam_mmd):
    model.train()
    it_tgt = iter(tgt_loader)
    cls_sum = coral_sum = mmd_sum = nstep = 0.0
    for Xs, ys in src_loader:
        try:
            Xt = next(it_tgt)
        except StopIteration:
            it_tgt = iter(tgt_loader); Xt = next(it_tgt)

        # 形が(B,C,T)想定。必要なら unsqueeze(1)
        Xs, ys, Xt = Xs.to(device), ys.to(device), Xt.to(device)

        Zs, log_s = model.forward_with_features(Xs)
        Zt, _     = model.forward_with_features(Xt)

        loss_cls   = ce(log_s, ys)
        loss_coral = coral_loss(Zs, Zt) if lam_coral>0 else 0.0
        loss_mmd   = mmd_rbf(Zs, Zt)   if lam_mmd  >0 else 0.0
        loss = loss_cls + lam_coral*loss_coral + lam_mmd*loss_mmd

        opt.zero_grad(); loss.backward(); opt.step()

        cls_sum   += float(loss_cls)
        coral_sum += float(loss_coral) if lam_coral>0 else 0.0
        mmd_sum   += float(loss_mmd) if lam_mmd>0 else 0.0
        nstep     += 1
    scheduler.step()
    print(f"{stage} | CE:{cls_sum/nstep:.4f}  CORAL:{coral_sum/max(1,nstep):.4f}  MMD:{mmd_sum/max(1,nstep):.4f}")

# ----- Stage A: CORALのみ（まず安定） -----
E_A = 30
for ep in range(E_A):
    one_epoch("A", lam_coral=1e-2, lam_mmd=0.0)
torch.save(model.state_dict(), r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageA_coral.pth")

# ----- Stage B: CORAL + MMD（細部を詰める） -----
E_B = 30  # 合計60ep
for ep in range(E_B):
    one_epoch("B", lam_coral=1e-2, lam_mmd=1e-2)
torch.save(model.state_dict(), r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_coral_mmd.pth")


A | CE:0.6266  CORAL:0.0078  MMD:0.0000
A | CE:0.5528  CORAL:0.0051  MMD:0.0000
A | CE:0.5296  CORAL:0.0033  MMD:0.0000
A | CE:0.5124  CORAL:0.0027  MMD:0.0000
A | CE:0.5045  CORAL:0.0019  MMD:0.0000
A | CE:0.4844  CORAL:0.0014  MMD:0.0000
A | CE:0.4779  CORAL:0.0015  MMD:0.0000
A | CE:0.4721  CORAL:0.0016  MMD:0.0000
A | CE:0.4741  CORAL:0.0011  MMD:0.0000
A | CE:0.4528  CORAL:0.0013  MMD:0.0000
A | CE:0.4643  CORAL:0.0013  MMD:0.0000
A | CE:0.4494  CORAL:0.0010  MMD:0.0000
A | CE:0.4435  CORAL:0.0009  MMD:0.0000
A | CE:0.4387  CORAL:0.0009  MMD:0.0000
A | CE:0.4471  CORAL:0.0009  MMD:0.0000
A | CE:0.4339  CORAL:0.0006  MMD:0.0000
A | CE:0.4346  CORAL:0.0005  MMD:0.0000
A | CE:0.4266  CORAL:0.0004  MMD:0.0000
A | CE:0.4333  CORAL:0.0004  MMD:0.0000
A | CE:0.4295  CORAL:0.0004  MMD:0.0000
A | CE:0.3977  CORAL:0.0004  MMD:0.0000
A | CE:0.3992  CORAL:0.0004  MMD:0.0000
A | CE:0.3822  CORAL:0.0004  MMD:0.0000
A | CE:0.3874  CORAL:0.0004  MMD:0.0000
A | CE:0.3982  CORAL:0.0004  MMD:0.0000


In [2622]:
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader

TGT_NPZ = r"E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz"
npz = np.load(TGT_NPZ)

# 保存済みのFTデータ（すでに x_all[ft_idx] 済み）
x_ft = torch.from_numpy(npz["x_ft"])   # (N_ft, C, T)
y_ft = torch.from_numpy(npz["y_ft"])   # (N_ft,)

# グローバル → ローカル変換に使う
ft_idx_global   = npz["ft_idx"]        # 長さ N_ft, x_all 上のインデックス（昇順のはず）
ft_train_global = npz["ft_train_idx"]  # これも x_all 上のインデックス
ft_val_global   = npz["ft_val_idx"]

# --- ここがポイント：global → local index に変換 ---
# ft_idx_global は sort 済みなので searchsorted が使える
ft_train_local = np.searchsorted(ft_idx_global, ft_train_global)
ft_val_local   = np.searchsorted(ft_idx_global, ft_val_global)

# 念のため範囲チェック
print("x_ft shape:", x_ft.shape)
print("max train_local:", ft_train_local.max(), "max val_local:", ft_val_local.max())

# これで x_ft / y_ft に安全に刺せる
x_tr, y_tr = x_ft[ft_train_local], y_ft[ft_train_local]
x_va, y_va = x_ft[ft_val_local],   y_ft[ft_val_local]

print("FT train:", x_tr.shape, "FT val:", x_va.shape)

ft_train_loader = DataLoader(TensorDataset(x_tr, y_tr),
                             batch_size=32, shuffle=True, drop_last=False)
ft_val_loader   = DataLoader(TensorDataset(x_va, y_va),
                             batch_size=64, shuffle=False, drop_last=False)


x_ft shape: torch.Size([30, 8, 1251])
max train_local: 29 max val_local: 24
FT train: torch.Size([24, 8, 1251]) FT val: torch.Size([6, 8, 1251])


In [2623]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Stage B のUDA済みモデルを読み込み
model = EEGNet(num_classes=2, channels=x_pool.shape[1], samples=x_pool.shape[2]).to(device)
stageB_ckpt = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_coral_mmd.pth"
model.load_state_dict(torch.load(stageB_ckpt, map_location=device))

# いったん全部凍結して…
for p in model.parameters():
    p.requires_grad = False

# 最終層（classify）だけ学習可にする
for p in model.classify.parameters():
    p.requires_grad = True

# Optimizer & Loss
opt = torch.optim.AdamW(model.classify.parameters(), lr=1e-2, weight_decay=1e-1)
ce  = torch.nn.CrossEntropyLoss()

# ★ここでエポック数を好きに決める
N_FT_EPOCHS = 7   # 例: 3エポック。増やしたければここを変える

for epoch in range(1, N_FT_EPOCHS + 1):
    # --- train ---
    model.train()
    tr_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in ft_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = ce(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

        tr_loss += float(loss)
        pred = logits.argmax(1)
        correct += (pred == yb).float().sum().item()
        total += len(yb)
    tr_loss /= len(ft_train_loader)
    tr_acc = correct / total if total > 0 else 0.0

    # --- val（モニタ用だけ。早期停止には使わない） ---
    model.eval()
    va_correct = 0
    va_total = 0
    with torch.no_grad():
        for xb, yb in ft_val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            va_correct += (pred == yb).float().sum().item()
            va_total += len(yb)
    va_acc = va_correct / va_total if va_total > 0 else 0.0

    print(f"[FT] epoch {epoch}  train_loss={tr_loss:.4f}  "
          f"train_acc={tr_acc*100:.1f}%  val_acc={va_acc*100:.1f}%")

# 最後のエポックのモデルを保存（必要なら）
best_path = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_afterFT.pth"
torch.save(model.state_dict(), best_path)
print(f"Saved FT model to: {best_path}")



device: cuda
[FT] epoch 1  train_loss=0.8301  train_acc=54.2%  val_acc=50.0%
[FT] epoch 2  train_loss=1.1018  train_acc=50.0%  val_acc=50.0%
[FT] epoch 3  train_loss=0.5704  train_acc=62.5%  val_acc=50.0%
[FT] epoch 4  train_loss=0.4675  train_acc=75.0%  val_acc=50.0%
[FT] epoch 5  train_loss=0.3407  train_acc=87.5%  val_acc=33.3%
[FT] epoch 6  train_loss=0.2872  train_acc=83.3%  val_acc=33.3%
[FT] epoch 7  train_loss=0.1862  train_acc=91.7%  val_acc=33.3%
Saved FT model to: C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_afterFT.pth


In [2624]:
import numpy as np, torch
from torch.utils.data import TensorDataset, DataLoader

TGT_NPZ = r"E:\EEG_B4\UDADataset\P001\target_P001_S002_prepared.npz"
npz = np.load(TGT_NPZ)

x_ft = torch.from_numpy(npz["x_ft"])   # (N_ft, C, T)
y_ft = torch.from_numpy(npz["y_ft"])

ft_idx_global   = npz["ft_idx"]
ft_train_global = npz["ft_train_idx"]
ft_val_global   = npz["ft_val_idx"]

# global → local 変換
ft_train_local = np.searchsorted(ft_idx_global, ft_train_global)
ft_val_local   = np.searchsorted(ft_idx_global, ft_val_global)

x_tr, y_tr = x_ft[ft_train_local], y_ft[ft_train_local]
x_va, y_va = x_ft[ft_val_local],   y_ft[ft_val_local]

print("FT train:", x_tr.shape, "FT val:", x_va.shape)

ft_train_loader = DataLoader(TensorDataset(x_tr, y_tr),
                             batch_size=32, shuffle=True, drop_last=False)
ft_val_loader   = DataLoader(TensorDataset(x_va, y_va),
                             batch_size=64, shuffle=False, drop_last=False)


FT train: torch.Size([24, 8, 1251]) FT val: torch.Size([6, 8, 1251])


In [2625]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

C, T = x_ft.shape[1], x_ft.shape[2]

# ★ ソースだけで学習した元モデルをロード
SRC_CKPT = r"C:\Users\kgtk0\Documents\Python\EEGNet_new\best.pt"
model_noUDA_FT = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_noUDA_FT.load_state_dict(torch.load(SRC_CKPT, map_location=device))

# いったん全部凍結
for p in model_noUDA_FT.parameters():
    p.requires_grad = False

# classify だけ学習可（No-UDA + FT）
for p in model_noUDA_FT.classify.parameters():
    p.requires_grad = True

opt = torch.optim.AdamW(model_noUDA_FT.classify.parameters(), lr=1e-2, weight_decay=1e-4)
ce  = torch.nn.CrossEntropyLoss()

N_FT_EPOCHS = 7
  # 好きに変えてOK

for epoch in range(1, N_FT_EPOCHS + 1):
    # --- train ---
    model_noUDA_FT.train()
    tr_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in ft_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model_noUDA_FT(xb)
        loss = ce(logits, yb)
        opt.zero_grad()
        loss.backward()
        # 勾配クリップ入れるならここ
        # torch.nn.utils.clip_grad_norm_(model_noUDA_FT.classify.parameters(), max_norm=5.0)
        opt.step()

        tr_loss += float(loss)
        pred = logits.argmax(1)
        correct += (pred == yb).float().sum().item()
        total += len(yb)
    tr_loss /= len(ft_train_loader)
    tr_acc = correct / total if total > 0 else 0.0

    # --- val（見るだけ） ---
    model_noUDA_FT.eval()
    va_correct = va_total = 0
    with torch.no_grad():
        for xb, yb in ft_val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model_noUDA_FT(xb).argmax(1)
            va_correct += (pred == yb).float().sum().item()
            va_total   += len(yb)
    va_acc = va_correct / va_total if va_total > 0 else 0.0

    print(f"[No-UDA+FT] epoch {epoch}  train_loss={tr_loss:.4f}  "
          f"train_acc={tr_acc*100:.1f}%  val_acc={va_acc*100:.1f}%")

# 最後のエポックを保存
ckpt_noUDA_FT = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_source_only_afterFT.pth"
torch.save(model_noUDA_FT.state_dict(), ckpt_noUDA_FT)
print("saved:", ckpt_noUDA_FT)


device: cuda
[No-UDA+FT] epoch 1  train_loss=0.9299  train_acc=50.0%  val_acc=50.0%
[No-UDA+FT] epoch 2  train_loss=0.6835  train_acc=58.3%  val_acc=50.0%
[No-UDA+FT] epoch 3  train_loss=0.6553  train_acc=54.2%  val_acc=50.0%
[No-UDA+FT] epoch 4  train_loss=0.3606  train_acc=75.0%  val_acc=50.0%
[No-UDA+FT] epoch 5  train_loss=0.3515  train_acc=83.3%  val_acc=50.0%
[No-UDA+FT] epoch 6  train_loss=0.2600  train_acc=83.3%  val_acc=50.0%
[No-UDA+FT] epoch 7  train_loss=0.3057  train_acc=79.2%  val_acc=33.3%
saved: C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_source_only_afterFT.pth


In [2626]:
from sklearn.metrics import f1_score, confusion_matrix

def eval_on_test(model, x_test, y_test, device):
    model.eval()
    X = x_test.to(device)
    Y = y_test.to(device)
    with torch.no_grad():
        logits = model(X)
        pred = logits.argmax(1)
    acc = (pred==Y).float().mean().item()
    f1  = f1_score(Y.cpu(), pred.cpu(), average="macro")
    cm  = confusion_matrix(Y.cpu(), pred.cpu())
    return acc, f1, cm

C, T = x_test.shape[1], x_test.shape[2]

# 1) No-UDA（ソースだけ）
model_src = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_src.load_state_dict(torch.load(SRC_CKPT, map_location=device))
acc0, f10, cm0 = eval_on_test(model_src, x_test, y_test, device)

# 2) No-UDA + FT
model_srcFT = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_srcFT.load_state_dict(torch.load(ckpt_noUDA_FT, map_location=device))
accFT0, f1FT0, cmFT0 = eval_on_test(model_srcFT, x_test, y_test, device)

# 3) UDA Stage B
stageB_ckpt = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_coral_mmd.pth"
model_stageB = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_stageB.load_state_dict(torch.load(stageB_ckpt, map_location=device))
accB, f1B, cmB = eval_on_test(model_stageB, x_test, y_test, device)

# 4) UDA + FT（StageB+FT）
ckpt_stageB_FT = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_afterFT.pth"
model_stageB_FT = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_stageB_FT.load_state_dict(torch.load(ckpt_stageB_FT, map_location=device))
accBFT, f1BFT, cmBFT = eval_on_test(model_stageB_FT, x_test, y_test, device)

print(f"[No-UDA]        Acc={acc0*100:.1f}%  F1={f10:.3f}\nCM=\n{cm0}\n")
print(f"[No-UDA + FT]   Acc={accFT0*100:.1f}%  F1={f1FT0:.3f}\nCM=\n{cmFT0}\n")
print(f"[Stage B]       Acc={accB*100:.1f}%  F1={f1B:.3f}\nCM=\n{cmB}\n")
print(f"[Stage B + FT]  Acc={accBFT*100:.1f}%  F1={f1BFT:.3f}\nCM=\n{cmBFT}\n")

print("========== Summary ==========")
print(f"No-UDA      : {acc0*100:5.1f}% (F1={f10:.3f})")
print(f"No-UDA + FT : {accFT0*100:5.1f}% (F1={f1FT0:.3f})")
print(f"Stage B     : {accB*100:5.1f}% (F1={f1B:.3f})")
print(f"StageB + FT : {accBFT*100:5.1f}% (F1={f1BFT:.3f})")


[No-UDA]        Acc=53.8%  F1=0.536
CM=
[[3 4]
 [2 4]]

[No-UDA + FT]   Acc=53.8%  F1=0.512
CM=
[[5 2]
 [4 2]]

[Stage B]       Acc=53.8%  F1=0.536
CM=
[[4 3]
 [3 3]]

[Stage B + FT]  Acc=61.5%  F1=0.606
CM=
[[5 2]
 [3 3]]

========== Summary ==========
No-UDA      :  53.8% (F1=0.536)
No-UDA + FT :  53.8% (F1=0.512)
Stage B     :  53.8% (F1=0.536)
StageB + FT :  61.5% (F1=0.606)


In [2627]:
C_S, T_S = x_train_src.shape[1], x_train_src.shape[2]
x_test_src_t = x_test_src.to(device)
y_test_src_t = y_test_src.to(device)

# 1) Physionetをもとのやつに入れる
model_src = EEGNet(num_classes=2, channels=C_S, samples=T_S).to(device)
model_src.load_state_dict(torch.load(SRC_CKPT, map_location=device))
acc0, f10, cm0 = eval_on_test(model_src, x_test_src_t, y_test_src_t, device)

# 2) NoUDA＋FTのモデルにぶっこむ
model_srcFT = EEGNet(num_classes=2, channels=C_S, samples=T_S).to(device)
model_srcFT.load_state_dict(torch.load(ckpt_noUDA_FT, map_location=device))
accFT0, f1FT0, cmFT0 = eval_on_test(model_srcFT, x_test_src_t, y_test_src_t, device)

# 3) UDA Stage Bにぶっこむ
stageB_ckpt = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_coral_mmd.pth"
model_stageB = EEGNet(num_classes=2, channels=C_S, samples=T_S).to(device)
model_stageB.load_state_dict(torch.load(stageB_ckpt, map_location=device))
accB, f1B, cmB = eval_on_test(model_stageB, x_test_src_t, y_test_src_t, device)

# 4) UDA + FT（StageB+FT）
ckpt_stageB_FT = r"C:\Users\kgtk0\Documents\Python\UDA_FT\P001\ckpt_stageB_afterFT.pth"
model_stageB_FT = EEGNet(num_classes=2, channels=C, samples=T).to(device)
model_stageB_FT.load_state_dict(torch.load(ckpt_stageB_FT, map_location=device))
accBFT, f1BFT, cmBFT = eval_on_test(model_stageB_FT, x_test_src_t, y_test_src_t, device)





print(f"[PhysioNet in-domain]        Acc={acc0*100:.1f}%  F1={f10:.3f}\nCM=\n{cm0}\n")
print(f"[PhysioNet NoUDA+FT]        Acc={accFT0*100:.1f}%  F1={f1FT0:.3f}\nCM=\n{cmFT0}\n")
print(f"[Physionet Stage B]       Acc={accB*100:.1f}%  F1={f1B:.3f}\nCM=\n{cmB}\n")
print(f"[Physionet Stage B + FT]  Acc={accBFT*100:.1f}%  F1={f1BFT:.3f}\nCM=\n{cmBFT}\n")

print("========== Summary ==========")
print(f"No-UDA      : {acc0*100:5.1f}% (F1={f10:.3f})")
print(f"No-UDA + FT : {accFT0*100:5.1f}% (F1={f1FT0:.3f})")
print(f"Stage B     : {accB*100:5.1f}% (F1={f1B:.3f})")
print(f"StageB + FT : {accBFT*100:5.1f}% (F1={f1BFT:.3f})")



[PhysioNet in-domain]        Acc=80.7%  F1=0.792
CM=
[[759 173]
 [ 98 375]]

[PhysioNet NoUDA+FT]        Acc=76.9%  F1=0.751
CM=
[[730 202]
 [122 351]]

[Physionet Stage B]       Acc=79.7%  F1=0.774
CM=
[[784 148]
 [137 336]]

[Physionet Stage B + FT]  Acc=78.9%  F1=0.761
CM=
[[797 135]
 [161 312]]

========== Summary ==========
No-UDA      :  80.7% (F1=0.792)
No-UDA + FT :  76.9% (F1=0.751)
Stage B     :  79.7% (F1=0.774)
StageB + FT :  78.9% (F1=0.761)
